# 数据简单探查

## 读取数据

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.pyplot import title
from numpy.f2py.crackfortran import ignorecontains
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder

from main2 import accuracy

# 注意：test.csv本身无Survived列，gender_submission.csv是提交示例，若仅做预测可不用y_test
train_df = pd.read_csv('train.csv')
x_test = pd.read_csv('test.csv')

# 若需评估，可将train_df拆分训练集和验证集，而非直接用gender_submission
# 此处暂时注释y_test相关，若有验证集需求可自行拆分
df = pd.read_csv('gender_submission.csv')
y_test = df['Survived']
y_train=train_df['Survived']



## 数据探查

In [ ]:
print('训练集大致情况:')
train_df.info()
print('测试集大致情况:')
x_test.info()


# 各特征与Survived关系并进行可视化

## embarked与survival关系

In [ ]:
import seaborn as sns
sns.barplot(x='Embarked', y='Survived', data=train_df,hue='Embarked',palette='muted')
plt.show()
#使用众数填充缺失值
print(train_df['Embarked'].value_counts())
train_df['Embarked']=train_df['Embarked'].fillna('S')


## sex与survival关系

In [ ]:
import seaborn as sns
sns.barplot(x='Sex', y='Survived', data=train_df,hue='Sex',palette='muted')
plt.show()

## Parch与survival关系

In [ ]:
import seaborn as sns
sns.barplot(x='Parch', y='Survived', data=train_df,hue='Parch',palette='muted')
plt.show()

## SibSp与survival关系

In [ ]:
import seaborn as sns
sns.barplot(x='SibSp', y='Survived', data=train_df,hue='SibSp',palette='muted')
plt.show()

## Pclass与survival关系

In [ ]:
import seaborn as sns
sns.barplot(x='Pclass', y='Survived', data=train_df,palette='muted')
plt.show()

## Fare与survival关系

In [ ]:
import seaborn as sns
agefacet=sns.FacetGrid(train_df,hue='Survived',aspect=3,palette='muted')
agefacet.map(sns.kdeplot,'Fare',fill=True)
agefacet.set(xlim=(0,512))
agefacet.add_legend()


## Age与survival关系

In [ ]:
#Age特征处理
#1、age与survived的关系
import seaborn as sns
agefacet=sns.FacetGrid(train_df,hue='Survived',aspect=3,palette='muted')
agefacet.map(sns.kdeplot,'Age',fill=True)
agefacet.set(xlim=(0,80))
agefacet.add_legend()

# 特征工程

## Cabin特征处理

### cabin填充缺失值

In [ ]:
x_test['Cabin']=x_test['Cabin'].fillna('U')
train_df['Cabin']=train_df['Cabin'].fillna('U')
#提取cabin首字母(船舱的类型）
x_test['Deck']=x_test['Cabin'].map(lambda x:x[0])
train_df['Deck']=train_df['Cabin'].map(lambda x:x[0])
#查看不同cabin首字母与survived关系
sns.barplot(x='Deck', y='Survived', data=train_df,palette='muted')

### 根据首字母创建新特征，并为其赋值（具有一定线性关系）

In [ ]:

#根据数值分布输出数字,
def classifier(x):
    known_decks=['U','C','E','G','D','A','B','F','T']
    if x not in known_decks:
        return -1       #防止测试集中出现未知值
    elif x in ['E','D','B']:
        return 2
    elif x in ['C','F']:
        return 1
    else:
        return 0

train_df['Deck']=train_df['Deck'].map(classifier).astype('int')
x_test['Deck']=x_test['Deck'].map(classifier).astype('int')
print(train_df['Deck'].value_counts())
print(x_test['Deck'].value_counts())

## 新建Famliy Size特征

In [ ]:
train_df['Family Size']=train_df['Parch']+train_df['SibSp']+1    #加一等于把自己算上
x_test['Family Size']=x_test['Parch']+x_test['SibSp']+1
#查看家庭成员数量与survived关系
sns.barplot(x='Family Size', y='Survived', data=train_df,palette='muted')
#根据家庭数目大小创建新特征与存活率成正相关(小（0),中（1），大（2）），避免数据维度过大
def family_size(x):
    konws_size=[1,2,3,4,5,6,7,8,11]
    if x not in konws_size:
        return -1
    elif x in [5,6,8,11]:
        return 0
    elif x in [1,7]:
        return 1
    else:
        return 2
    return family_size(x)
train_df['Family Size']=train_df['Family Size'].map(family_size)
x_test['Family Size']=x_test['Family Size'].map(family_size)
print(train_df['Family Size'].value_counts())
print(x_test['Family Size'].value_counts())

### Fare缺失值处理

## Fare特征处理

In [ ]:
#fare缺失值过小,查看与survived关系
#先对fare做初步数据分箱后绘图(多试几次划分数量以确定更清晰的梯度差异）
train_df['Fare_bins']=pd.cut(train_df['Fare'],bins=3)
x_test['Fare_bins']=pd.cut(x_test['Fare'],bins=3)
sns.barplot(x='Fare_bins', y='Survived', data=train_df,palette='muted')



In [ ]:
#查看缺失个体相关的信息
display(train_df[train_df['Fare'].isnull()])
display(x_test[x_test['Fare'].isnull()])
#利用船舱等级，登船口，性别，仓位未知的旅客的平均票价进行填充
price=train_df[(train_df['Pclass']==3)&(train_df['Embarked']=='S')&(train_df['Sex']=='male')&(train_df['Cabin']=='U')]['Fare'].mean()
x_test['Fare']=x_test['Fare'].fillna(price)


### Fare特征缩放

In [ ]:
#根据大致分布分为3部分，低消费（0），中消费（1），高消费（2）（与存活率成正相关）
def fare_size(x):
    if x<170.776:
        return 0
    elif 170.776<=x<=341.553:
        return 1
    else:
        return 2
    return fare_size(x)
train_df['Fare']=train_df['Fare'].map(fare_size).astype('int')
x_test['Fare']=x_test['Fare'].map(fare_size).astype('int')

## Ticket特征处理

### Ticket特征处理

In [ ]:
#提取各票号的乘客数量
Ticket_count=train_df['Ticket'].value_counts()
Ticket_count.head(5)

### 创建新特征TicketCom

In [ ]:
#根据相同票号的乘客数量创建新特征
train_df['TicketCom']=train_df['Ticket'].map(Ticket_count)
#测试集映射，用1填充训练集未出现的票号，表示独票
x_test['TicketCom']=x_test['Ticket'].map(Ticket_count).fillna(1)
#查看新建特征与survival关系，方便后续缩放
sns.barplot(x='TicketCom', y='Survived', data=train_df,palette='muted')


### TicketCom特征缩放

In [ ]:
#根据生还率缩放数据，使其具有一定线性关系
def ticket_size(x):
    if x in [1,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]:  #极低生还
        return 0
    elif x in [2,4]:    #中生还率
        return 1
    else:                #高生还率
        return 2
    return ticket_size(x)
train_df['TicketCom']=train_df['TicketCom'].map(ticket_size)
x_test['TicketCom']=x_test['TicketCom'].map(ticket_size)
print(train_df['TicketCom'].value_counts())

## Name特征处理

In [ ]:
#Name特征处理
#取出头衔
train_df['Title']=train_df['Name'].map(lambda x:x.split(',')[1].split('.')[0].strip())
x_test['Title']=x_test['Name'].map(lambda x:x.split(',')[1].split('.')[0].strip())
print(train_df['Title'].value_counts())
#查看头衔与存活率关系
sns.barplot(x='Title', y='Survived', data=train_df,palette='muted')

In [ ]:
#头衔名称较长且数量多，旋转标签避免重叠，提升可读性
sns.barplot(x='Title', y='Survived', data=train_df,palette='muted')
plt.xticks(rotation=45, ha='right')     #右转45度，右对齐
plt.tight_layout()      #自动调整布局，防止标签别截断
plt.show()

In [ ]:
#根据存活率重新进行分类
def title_classifier(x):
    konws_title=['Mr','Mrs','Miss','Master','Dr','Rev','Major','Col','Mlle','Mme','Ms','Don','Jonkheer','Lady','Sir','the Countess','Capt']
    if x not in konws_title:
        return -1           #避免未知值
    elif x in ['Mr','Don','Rev','Capt','Jonkheer']:   #极低生还率
        return 0
    elif x in ['Dr','Major','Col','Master']:         #较低生还率
        return 1
    elif x in ['Mrs','Miss']:                  #较高生还率
        return 2
    else:                                       #极高生还率
        return 3
    return title_classifier(x)
train_df['Title']=train_df['Title'].map(title_classifier)
x_test['Title']=x_test['Title'].map(title_classifier).fillna(1) #防止未知报错
print('训练集处理结果\n',train_df['Title'].value_counts())
print('测试集处理结果\n',x_test['Title'].value_counts())


## Age特征处理

### Age缺失值处理

In [ ]:

from sklearn.preprocessing import OneHotEncoder
import  pandas as pd
#删除无用特征,将字符串进行独热编码
#训练集独热编码
train_df.info()
train_df=train_df.drop(['PassengerId','Name','SibSp','Parch','Cabin','Ticket','Fare_bins'],axis=1)
ohe=OneHotEncoder()
df1=train_df[['Sex','Embarked']]
df1=ohe.fit_transform(df1).toarray()
df1=pd.DataFrame(df1,columns=ohe.get_feature_names_out())
train_df=train_df.drop(['Sex','Embarked'],axis=1)
train_df=pd.concat([train_df,df1],axis=1)
#测试集独热编码
x_test.info()
x_test=x_test.drop(['PassengerId','Name','SibSp','Parch','Cabin','Ticket','Fare_bins'],axis=1)
ohe=OneHotEncoder()
df2=x_test[['Sex','Embarked']]
df2=ohe.fit_transform(df2).toarray()
df2=pd.DataFrame(df2,columns=ohe.get_feature_names_out())
x_test=x_test.drop(['Sex','Embarked'],axis=1)
x_test=pd.concat([x_test,df2],axis=1)
# print('测试集大致情况：')
# train_df.info()
# print('测试集大致情况：')
# x_test.info()

In [ ]:
#选出与age有关的特征
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
train_df_age_know=train_df[train_df['Age'].notnull()]
train_x=train_df_age_know.drop(['Age'],axis=1)
train_y=train_df_age_know['Age']
selector=SelectKBest(f_classif,k=5)
selector.fit_transform(train_x,train_y)
best_canshu=train_x.columns[selector.get_support()]
best_canshu


In [ ]:
#重新划分数据集
train_df1=train_df[['Pclass', 'Deck', 'TicketCom', 'Embarked_C','Age']]
x_test1=x_test[['Pclass', 'Deck', 'TicketCom', 'Embarked_C','Age']]
train_df_age_know=train_df1[train_df1['Age'].notnull()]
#训练集
train_x1=train_df_age_know.drop(['Age'],axis=1)
train_y1=train_df_age_know['Age']
#两个测试集
train_df_age_unknow=train_df1[train_df1['Age'].isnull()]
test_x1=train_df_age_unknow.drop(['Age'],axis=1)
x_test_age=x_test1[x_test1['Age'].isnull()]
test_x2=x_test_age.drop(['Age'],axis=1)

In [ ]:
#构建随机森林模型，填充Age缺失值
from sklearn.ensemble import RandomForestRegressor
rf_model=RandomForestRegressor(random_state=42)
rf_model.fit(train_x1,train_y1)
y1=rf_model.predict(test_x1)
y2=rf_model.predict(test_x2)
#将预测值按顺序填入原数据中
train_df.loc[train_df['Age'].isnull(),'Age']=y1
x_test.loc[x_test['Age'].isnull(),'Age']=y2
print('测试集大致情况：')
train_df.info()
print('测试集大致情况：')
x_test.info()
print('特征工程正式完成！！！')

# 构建模型进行预测目标值

## 先用随机搜索确定参数大致范围

In [ ]:
#划分标签库
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
#正式划分数据集
x_train=train_df.drop(['Survived'],axis=1)
y_train=train_df['Survived']

#利用随机搜索确定参数大致范围
random_grid={'n_estimators': range(50,400,50),
    'max_depth': [None,5,10,15,20],
    'min_samples_split':range(2,10),
    'min_samples_leaf': range(1,5),
    'criterion': ['gini', 'entropy']}
random_search=RandomizedSearchCV(estimator=RandomForestClassifier(),
                   param_distributions=random_grid,
                   n_iter=100,
                   cv=5,
                   random_state=42,
                   scoring='accuracy',
                   n_jobs=-1)
random_search.fit(x_train,y_train)
#查看五组最佳参数
topfive_params=random_search.cv_results_['params'][:5]
print('查看五组最佳参数：\n',topfive_params)
print('查看五组最佳参数得分：\n',random_search.cv_results_['mean_test_score'][:5])
#进行可视化，创建DataFrame
df=pd.DataFrame({'params':topfive_params,'score':random_search.cv_results_['mean_test_score'][:5]})
print('查看可视化结果：\n',df)


## 进行分区偶合调优进一步确定参数

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
param_grid={'n_estimators':[150],
    'max_depth': [5],
    'min_samples_split':range(2,10) ,
    'min_samples_leaf': range(1,5) ,
    'criterion': ['gini', 'entropy']}
grid_search=GridSearchCV(estimator=RandomForestClassifier(),
                         param_grid=param_grid,
                         cv=5,
                         scoring='accuracy',
                         n_jobs=-1)
grid_search.fit(x_train,y_train)
print('查看超参数调优结果：\n',grid_search.best_params_)
print('查看超参数调优得分：\n',grid_search.best_score_)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {'n_estimators':range(200,300,20) ,
              'max_depth': [None,5,10,15,20,25,30],
              'min_samples_split':[6],
              'min_samples_leaf':[1],
              'criterion':['entropy']}
grid_search = GridSearchCV(estimator=RandomForestClassifier(),
                           param_grid=param_grid,
                           cv=5,
                           scoring='accuracy',
                           n_jobs=-1)
grid_search.fit(x_train, y_train)
print('查看超参数调优结果：\n', grid_search.best_params_)
print('查看超参数调优得分：\n', grid_search.best_score_)


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import joblib
param_grid = {'n_estimators':[260] ,
              'max_depth': [5],
              'min_samples_split':[7,8,9,10,11],
              'min_samples_leaf':[2,3,4,5],
              'criterion':['entropy','gini']}
grid_search = GridSearchCV(estimator=RandomForestClassifier(),
                           param_grid=param_grid,
                           cv=5,
                           scoring='accuracy',
                           n_jobs=-1)
grid_search.fit(x_train, y_train)
print('查看超参数调优结果：\n', grid_search.best_params_)
print('查看超参数调优得分：\n', grid_search.best_score_)

In [ ]:
#保存最佳参数
import joblib
from sklearn.ensemble import RandomForestClassifier
model=RandomForestClassifier(n_estimators=150,
                             max_depth=5,
                             min_samples_split=6,
                             min_samples_leaf=1,
                             criterion='entropy')
joblib.dump(model,'rf_model.pkl')

## 使用模型进行预测

In [ ]:
import joblib
from sklearn.metrics import classification_report
model=joblib.load('rf_model.pkl')
predict=model.predict(x_test)
accuracy=accuracy_score(y_test,predict)
print('查看预测结果：\n',predict)
print('查看准确率：\n',accuracy)
print('查看预测结果：\n',classification_report(y_test,predict))
#保存预测结果
import pandas as pd
x_new_test = pd.read_csv('test.csv')
accuracy = accuracy_score(y_test, predict)
print('查看准确率：\n', accuracy)
result = pd.DataFrame({
    'PassengerId': x_new_test['PassengerId'],
    'Survived': predict.astype(int)         #确保为整数
})
result.to_csv('tmd别给劳资打低分.csv', index=False)